# 31b_v3

Writes `submission.zip` for the Gemma 4 Developer Agent competition. Run on CPU with internet off, then Submit.

In [ ]:
import json, zipfile
from pathlib import Path

FILES = json.loads("{\n \"agent.yaml\": \"name: swe_agent_v3\\nmodel: gemma-4-31b-it-qat-w4a16-ct\\n# v3 = v2 (baseline tools + step-by-step prompt) with 40 tool calls instead of 28, same 8-minute clock.\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\n  - get_code_neighbors\\n  - search_similar_code\\n  - get_code_subgraph\\n  - agent_tool:\\n      config_path: sub_agents/code_analyzer.yaml\\n      skip_summarization: true\\ngenerate_content_config: !include configs/sampling.yaml\\n\",\n \"configs/sampling.yaml\": \"temperature: 0.2\\ntop_p: 0.95\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 4096\\n  include_thoughts: true\\n\",\n \"eval_config.yaml\": \"# Per-task budgets. The scorer runs the ~120 hidden tasks ONE AT A TIME and the whole run must end within 12 hours\\n# (going over errors the submission), on 4x L4 where 31B is 2-3x slower than on our H200.\\n# v3 keeps the 8-minute clock (same worst-case run time as v2, which finished) and only raises the tool-call\\n# cap 28 -> 40, so a task that is still making progress is stopped by the clock, not the counter.\\nevaluation:\\n  timeout_seconds: 240\\n  max_tool_calls: 40\\n  max_time_minutes: 8\\n  max_turns: 120\\n\",\n \"prompts/analyzer.md\": \"You are a specialized code analysis sub-agent. Your role is to inspect repository source files, trace symbol relationships, and identify the root cause of the reported issue.\\n\\n## Instructions\\n1. Use `search_similar_code`, `get_code_neighbors`, `get_code_subgraph`, and `read_file` to locate the exact functions, classes, and line ranges involved in the bug.\\n2. Provide a concise, structured report containing:\\n   - Exact file paths and line numbers to modify.\\n   - Root cause explanation.\\n   - Recommended minimal code change.\\n\",\n \"prompts/system.md\": \"You are an expert software engineer fixing one reported issue in a Python repository at `/workspace`.\\nHidden tests, written by the project's maintainers for this issue, will check your fix. They test the\\nbehaviour the issue describes, using the exact names, arguments, messages and return values it mentions.\\n\\n## Your budget: 40 tool calls and 8 minutes. Plan them.\\n`submit_patch` and `get_status` are free. Every other tool call costs one. The clock usually runs out\\nbefore the calls do, so move quickly: short commands, no re-reading the same code.\\n\\n| Calls | Step |\\n|-------|------|\\n| 1-8   | LOCATE the code that causes the issue |\\n| 9-10  | REPRODUCE the issue with a short script |\\n| by 14 | EDIT: your first fix must be in place by call 14 at the latest |\\n| 15-34 | VERIFY and refine: re-run the script, run the related existing tests, fix what your change broke |\\n| last  | SUBMIT: call `submit_patch`, then reply with a two-line summary |\\n\\nNever spend the whole budget reading. A reasonable fix that is submitted beats a perfect one that is not.\\nDo not stop early either: after your first edit, always verify it before submitting.\\n\\n## 1. LOCATE\\n- Pull every concrete name out of the issue: functions, classes, parameters, error messages, CLI options.\\n- Start with `search_similar_code`, giving it a short description of the behaviour or a phrase from the issue.\\n  It searches the whole code index and returns the most relevant functions with their files and lines.\\n- Then use `get_code_neighbors` on the function you found to see its callers and callees, or\\n  `get_code_subgraph` to see how a few related functions connect.\\n- For an exact name, grep: `grep -rn \\\"name_from_issue\\\" --include=*.py . | grep -v \\\"/tests/\\\" | head -40`\\n- Read a function with `read_file` and `start_line`/`end_line` (at most 150 lines per call).\\n- Find how similar features are already done in the codebase and follow the same pattern.\\n- Find the existing tests for this code: `grep -rln \\\"function_name\\\" tests/ | head`.\\n- If the cause is still unclear after a few calls, ask the `code_analyzer_agent` once for the exact\\n  file, lines and root cause. Its tool calls count against your budget, so do not call it more than once.\\n\\n## 2. REPRODUCE (no files needed)\\nRun the example from the issue straight from stdin, so nothing is added to the repository:\\n```\\npython3 - <<'PY'\\nfrom package.module import thing\\nprint(thing(...))        # show the wrong behaviour from the issue\\nPY\\n```\\nIf it does not reproduce, re-read the issue: you may be looking at the wrong code path.\\n\\n## 3. EDIT\\n- Fix the root cause in the library source, not the symptom. Make the fix general, not specific to the example.\\n- Use the exact names, signatures, defaults, error types and messages that the issue asks for.\\n- If you add or change a parameter, update every place that needs it (callers, overloads, sync and async\\n  versions, type hints, `__all__`). Use grep to find them.\\n- Use `edit_file` with 2-4 lines of unique surrounding context in `old_string`. Keep the change small;\\n  do not reformat or refactor unrelated code.\\n- Never edit, add or delete test files (`tests/`, `test_*.py`, `conftest.py`): such changes are discarded.\\n\\n## 4. VERIFY\\n- Re-run your reproduction script: the output must now match what the issue expects.\\n- Run only the related test file, never the whole suite:\\n  `python3 -m pytest tests/test_x.py -q -x 2>&1 | tail -30`\\n- If a test fails because of your change, fix the source. If it also failed before your change (missing\\n  fixtures, network), ignore it. All dependencies are already installed and there is no network: never pip install.\\n- Check edge cases the issue mentions (empty input, None, different types).\\n\\n## 5. SUBMIT\\nCall `submit_patch`, check that `files_changed` is at least 1, then reply with a short text summary of the fix.\\nIf the budget is nearly used up (check with the free `get_status`), submit what you have.\\n\\n## Output limits\\nCommand output is truncated: always cut long output with `| head -50` or `| tail -30`.\\n\",\n \"sub_agents/code_analyzer.yaml\": \"name: code_analyzer_agent\\ndescription: Analyzes repository source files and symbol graphs to locate root causes.\\nmodel: gemma-4-31b-it-qat-w4a16-ct\\ninstruction: !include ../prompts/analyzer.md\\ntools:\\n  - read_file\\n  - search_similar_code\\n  - get_code_neighbors\\n  - get_code_subgraph\\ngenerate_content_config: !include ../configs/sampling.yaml\\n\"\n}")

out = Path('/kaggle/working/submission.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name, text in FILES.items():
        zf.writestr(name, text)
print(out, out.stat().st_size, 'bytes:', sorted(FILES))
print(FILES['agent.yaml'])
print(FILES['eval_config.yaml'])
